In [ ]:
import qdrant_client
from llama_index.core import SimpleDirectoryReader, Settings
from llama_index.core.indices import MultiModalVectorStoreIndex
from llama_index.core.storage import StorageContext
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.clip import ClipEmbedding
import time 
start_time = time.time()
clip_model = ClipEmbedding()
Settings.embed_model = clip_model
client = qdrant_client.QdrantClient(location=":memory:")
image_store = QdrantVectorStore(
    client=client,
    collection_name="lung_tumor_dataset"
)
storage_context = StorageContext.from_defaults(image_store=image_store)

data_folder_path = "./data"
start_time = time.time()
image_documents = SimpleDirectoryReader(input_dir=data_folder_path).load_data()
load_time = time.time() - start_time

if not image_documents:
    import sys
    sys.exit()
else:
    print(f"Successfully loaded {len(image_documents)} images as documents!")

start_time = time.time()
index = MultiModalVectorStoreIndex.from_documents(
    image_documents,
    storage_context=storage_context,
    image_embed_model=clip_model
)
encoding_time = time.time() - start_time

# ==========================================
# TEST RETRIEVAL across the dataset
# ==========================================
# Right now, your database has 8 entries. When you search, Qdrant will compare
# your query against all 8 vectors to find the closest matches.

# Since they are all similar slides, let's look at the Top 3 visually distinct matches.
top_k = 3
retriever = index.as_retriever(
    similarity_top_k=top_k,
    image_similarity_top_k=top_k
)

# Use a specific biological query related to the dataset to see if it makes sense.
# (CLIP translates this text into math, Qdrant finds the closest image math)
search_query = "An H&E stained biological tissue slide with granular, purplish cellular structures scattered within it."
start_time = time.time()
retrieval_results = retriever.retrieve(search_query)
search_time = time.time() - start_time

print(f"\n--- RETRIEVAL RESULTS (Found in {search_time:.4f} seconds) ---")
for i, res in enumerate(retrieval_results):
    print(f"Rank {i+1}:")
    print(f"   Found File: {res.node.metadata['file_name']}")
    print(f"   Mathematical Similarity Score: {res.score:.4f}")
    print("-" * 25)

Successfully loaded 9 images as documents!

--- RETRIEVAL RESULTS (Found in 0.0987 seconds) ---
Rank 1:
   Found File: tissue_hires_image_lung_6_tumor.png
   Mathematical Similarity Score: 0.3619
-------------------------
Rank 2:
   Found File: tissue_hires_image_lung_8_tumor.png
   Mathematical Similarity Score: 0.3480
-------------------------
Rank 3:
   Found File: tissue_hires_image_lung_1_tumor.png
   Mathematical Similarity Score: 0.3415
-------------------------
